# Error Analysis of Culture Medium Cleaning Pipeline

## Objective

This notebook evaluates the output of the current LaTeX cleaning pipeline.

The goals are:

1. Apply the existing cleaning functions to the JCM medium dataset.
2. Identify remaining formatting and parsing errors.
3. Group errors into common patterns.
4. Design additional cleaning strategies if necessary.

In [35]:
from google.colab import files
uploaded = files.upload()

Saving cleaning.py to cleaning.py


In [4]:
from google.colab import files
uploaded = files.upload()

Saving JCM_mediumID_mediumName.csv to JCM_mediumID_mediumName.csv


In [55]:
import cleaning
import importlib
importlib.reload(cleaning)

<module 'cleaning' from '/content/cleaning.py'>

In [56]:
import re
import cleaning
import pandas as pd
from typing import Any # The typing module helps programmers describe what kinds of data functions expect.
from cleaning import clean_df, verify, print_verification
df = pd.read_csv("JCM_mediumID_mediumName.csv")
df.head()
cleaned_df = clean_df(df)
print((cleaned_df.head()))
cleaned_df[["tex_text","tex_text_clean"]].head(5)
cleaned_df.to_csv("clean_df.csv", index = False)
print_verification(cleaned_df) #please note this result changed because i made some changes in cleaning files before the errors were more

   grmd                                md_name  \
0  1436               LIMNOCHORDIA L945 MEDIUM   
1  1479               SOLIDESULFOVIBRIO MEDIUM   
2  1481              Bold's Basal Medium (BBM)   
3  1477                GIBBONS MODIFIED MEDIUM   
4  1475  DESULFOBACTER MEDIUM WITH HORSE SERUM   

                                            tex_text  \
0  \mono{KH$_2$PO$_4$} {0.2} {g}\n\mono{MgCl$_2$$...   
1  \mono{KH$_2$PO$_4$}                           ...   
2  \mono{Agar}{20g}\\mono{ Distilled water}{980mL...   
3  \mono{Casamino acids (BD-Difco)}              ...   
4  \mono{Na$_2$SO$_4$} 　　　　                      ...   

                                      tex_text_clean  
0  \mono{KH$_2$PO$_4$} {0.2} {g}\n\mono{MgCl$_2$·...  
1  \mono{KH$_2$PO$_4$} {0.20}{g}\n\mono{NH$_4$Cl}...  
2  \mono{Agar} {20} {g}\n\mono{Distilled water} {...  
3  \mono{Casamino acids (BD-Difco)} {10.0}{g}\n\m...  
4  \mono{Na$_2$SO$_4$} {3.0}{g}\n\mono{KH$_2$PO$_...  
=== CLEANING VERIFICATION ===

 

# Remaining Unparseable
\mono Patterns

In [6]:
report = verify(cleaned_df)

errors = report["_unparseable_lines"]

len(errors)
errors[:10]

['\\mono{{N}-Acetyl-D-glucosamine} {1.0}{g}',
 '\\mono{{p}-Aminobenzoic acid} {50.0}{mg}',
 '\\mono{{p}-Aminobenzoic acid} {5.0}{mg}',
 '\\mono{{myo}-Inositol} {5.0}{mg}',
 '\\mono{{p}-Aminobenzoic acid} {5.0}{mg}',
 '\\mono{2.5% {N}-Acetyl-D-glucosamine solution} {20.0}{ml}',
 '\\mono{{p}-Aminobenzoic acid} {100.0}{mg}',
 '\\mono{{p}-Aminobenzoic acid} {50.0}{mg}',
 '\\mono{{n}-Butyric acid} {0.4}{ml}',
 '\\mono{{iso}-Butyric acid} {0.4}{ml}']

## Pattern 1: Extra braces in component names

Investigate lines containing double curly braces inside \mono names.

In [7]:
double_brace_errors = [
    line for line in errors
    if "{{" in line
]
len(double_brace_errors)
double_brace_errors

['\\mono{{N}-Acetyl-D-glucosamine} {1.0}{g}',
 '\\mono{{p}-Aminobenzoic acid} {50.0}{mg}',
 '\\mono{{p}-Aminobenzoic acid} {5.0}{mg}',
 '\\mono{{myo}-Inositol} {5.0}{mg}',
 '\\mono{{p}-Aminobenzoic acid} {5.0}{mg}',
 '\\mono{{p}-Aminobenzoic acid} {100.0}{mg}',
 '\\mono{{p}-Aminobenzoic acid} {50.0}{mg}',
 '\\mono{{n}-Butyric acid} {0.4}{ml}',
 '\\mono{{iso}-Butyric acid} {0.4}{ml}',
 '\\mono{{n}-Valeric acid} {0.2}{ml}',
 '\\mono{{iso}-Valeric acid} {0.2}{ml}',
 '\\mono{{p}-Aminobenzoic acid} {5.0}        {mg}',
 '\\mono{{p-}Aminobenzoic acid} {0.25}{mg}',
 '\\mono{{p}-Aminobenzoic acid} {5.0}{mg}',
 '\\mono{{p}-Aminobenzoic acid} {10.0}{mg}',
 '\\mono{{n}--Valeric acid} {1.0}{ml}',
 '\\mono{{iso}--Valeric acid} {1.0}{ml}',
 '\\mono{{iso}--Butyric acid} {1.0}{ml}',
 '\\mono{{p}--Aminobenzoic acid} {15.0}{mg}',
 '\\mono{{p}-Aminobenzoic acid} {100.0}{mg}',
 '\\mono{{p}--Aminobenzoic acid} {5.0}{mg}',
 '\\mono{{p}--Aminobenzoic acid} {80.0}{mg}',
 '\\mono{{N}--Acetyl--D--glucosamine (Si

## Analysis of Double-Brace Errors

The verification report identified 78 unparseable `\mono` lines.

To understand the cause, the errors containing `{{` were extracted and analysed.

The extracted prefixes were counted using Python's `Counter` class.

Results:

- p : 53 occurrences
- iso : 7 occurrences
- n : 6 occurrences
- myo : 5 occurrences
- N : 4 occurrences
- p- : 1 occurrence

This indicates that the majority of the remaining parsing failures are caused by LaTeX-style prefix formatting inside ingredient names rather than completely corrupted records.

In [12]:
len(double_brace_errors)
from collections import Counter
prefixes = []
for line in double_brace_errors:
  if "{{" in line:
    prefixes.append(line.split("{{")[1].split("}")[0])
Counter(prefixes)

Counter({'N': 4, 'p': 53, 'myo': 5, 'n': 6, 'iso': 7, 'p-': 1})

In [13]:
import pandas as pd
from collections import Counter

prefix_counts = Counter(prefixes)

summary = pd.DataFrame(
    prefix_counts.items(),
    columns=["Prefix", "Count"]
)

summary = summary.sort_values("Count", ascending=False)

summary

,Prefix,Count
1,p,53
4,iso,7
3,n,6
2,myo,5
0,N,4
5,p-,1


To see if all all 78 unparseable lines caused only by the double-brace prefix pattern, or are there other kinds of unparseable lines mixed in i ran the following codes below.

In [24]:
#len(double_brace_errors)
len(errors)

78

Two remaining verification failures are caused by valid chemical notation ({N}) inside ingredient names rather than malformed records. The current verification regular expression does not recognise this notation as valid.

In [26]:
other_errors = [
    line for line in errors
    if "{{" not in line
]

other_errors

['\\mono{2.5% {N}-Acetyl-D-glucosamine solution} {20.0}{ml}',
 '\\mono{10% {N}-Acetyl-D-glucosamine solution*} {10.0}{ml}']

## Analysis of Remaining Parsing Errors (Summary report)

The cleaning pipeline reported 78 unparseable `\mono` lines.

To investigate the cause, the errors were grouped by common formatting patterns.

### Findings

- Total unparseable lines: **78**
- Lines containing double opening braces (`{{`): **76**
- Remaining lines: **2**

The 76 double-brace cases mainly occur in chemical names containing prefixes such as:

- p-
- iso-
- n-
- myo-
- N-

These prefixes appear to originate from LaTeX formatting and interfere with the parser.

The remaining two lines contain valid chemical notation (`{N}`) inside the ingredient name and may represent a limitation of the verification regex rather than an actual data error.

Testing the functions i am trying to build to solve the above problem

In [28]:
import re

inner = []

for line in double_brace_errors:
    m = re.search(r"\{\{([^}]+)\}", line)
    if m:
        inner.append(m.group(1))

sorted(set(inner))

['N', 'iso', 'myo', 'n', 'p', 'p-']

In [31]:
import re

example = r"\mono{{iso}-Butyric acid} {0.4}{ml}"

fixed = re.sub(
    r"\{\{([^}]+)\}",
    r"{\1",
    example
)

print(example)
print(fixed)

\mono{{iso}-Butyric acid} {0.4}{ml}
\mono{iso-Butyric acid} {0.4}{ml}


In [58]:
from cleaning import clean_df, print_verification, verify
cleaned_df = clean_df(df)

print_verification(cleaned_df)

=== CLEANING VERIFICATION ===

  sfi                          0
  hspace                       0
  Mix_tag                      0
  mu_tag                       0
  cdot_tag                     0
  double_curly                 0
  double_backslash             0
  corrupted_v                  0
  unclosed_amount              0
  unit_mL                      0
  unit_vg                      0
  html_entity                  0
  plain_reference              0
  mono_valid_inner_braces      79
  mono_unparseable             2  <-- CHECK

Unparseable \mono lines (2):
  '\\mono{2.5% {N}-Acetyl-D-glucosamine solution} {20.0}{ml}'
  '\\mono{10% {N}-Acetyl-D-glucosamine solution*} {10.0}{ml}'


## After Prefix Brace Repair

A new cleaning rule (`pass4_prefix_braces`) was introduced to remove unnecessary LaTeX braces around chemical prefixes.

Before:

- double_curly errors: 61
- mono_unparseable: 78

After:

- double_curly errors: 0
- mono_unparseable: 2

Remaining two cases:

\mono{2.5% {N}-Acetyl-D-glucosamine solution} {20.0}{ml}

\mono{10% {N}-Acetyl-D-glucosamine solution*} {10.0}{ml}

These are not malformed ingredient names. The `{N}` notation is valid chemical naming. The remaining issue appears to originate from the verification pattern not supporting nested braces inside ingredient names.

In [64]:
cleaned_df = clean_df(df)
report = verify(cleaned_df)
errors = report["_unparseable_lines"]

len(errors)
errors

['\\mono{2.5% {N}-Acetyl-D-glucosamine solution} {20.0}{ml}',
 '\\mono{10% {N}-Acetyl-D-glucosamine solution*} {10.0}{ml}']

## Verification issue: nested braces

After introducing `pass4_prefix_braces`, the cleaning pipeline reduced:

- double curly errors: 61 → 0
- unparseable mono lines: 78 → 2

The remaining two lines are not corrupted data. They contain valid chemical notation:

\{N\}-Acetyl-D-glucosamine

The current verification regex cannot handle nested braces inside ingredient names. Therefore, the next step is to replace the regex-based verification with a brace-aware parser.

In [68]:
report = verify(cleaned_df)
errors = report["_unparseable_lines"]

In [74]:
check_mono_structure

<function __main__.check_mono_structure(line)>